# **Laboratorio 9: Airflow 🛫**

<center><strong>MDS7202: Laboratorio de Programación Científica para Ciencia de Datos - Primavera 2025</strong></center>

### Cuerpo Docente:

- Profesores: Diego Cortez, Gabriel Iturra
- Auxiliares: Melanie Peña, Valentina Rojas
- Ayudantes: Nicolás Cabello, Cristopher Urbina

### Equipo: SUPER IMPORTANTE - notebooks sin nombre no serán revisados

- Nombre de alumno 1: Javier Pinochet
- Nombre de alumno 2: Daniel Muñoz

### **Link de repositorio de GitHub:** [Insertar Repositorio](https://github.com/danmc899/MDS7202)

## Temas a tratar

- Construcción de pipelines productivos usando `Airflow`.

## Reglas:

- **Grupos de 2 personas**
- Fecha de entrega: Entregas Martes a las 23:59.
- Instrucciones del lab el viernes a las 16:15 en formato online. Asistencia no es obligatoria, pero se recomienda **fuertemente** asistir.
- <u>Prohibidas las copias</u>. Cualquier intento de copia será debidamente penalizado con el reglamento de la escuela.
- Tienen que subir el laboratorio a u-cursos y a su repositorio de github. Labs que no estén en u-cursos no serán revisados. Recuerden que el repositorio también tiene nota.
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente serán respondidos por este medio.
- Pueden usar cualquier material del curso que estimen conveniente.

### Objetivos principales del laboratorio

- Reconocer los componentes pricipales de `Airflow` y su funcionamiento.
- Poner en práctica la construcción de pipelines de `Airflow`.
- Automatizar procesos típicos de un proyecto de ciencia de datos mediante `Airflow` y `Docker`.

El laboratorio deberá ser desarrollado sin el uso indiscriminado de iteradores nativos de python (aka "for", "while"). La idea es que aprendan a exprimir al máximo las funciones optimizadas que nos entrega `pandas`, las cuales vale mencionar, son bastante más eficientes que los iteradores nativos sobre DataFrames.

# **Introducción**

<p align="center">
  <img src="https://media.tenor.com/OBQ6niqbxswAAAAM/legallyblonde.gif" width="300">
</p>

Vale, una estudiante del Magíster en Ciencia de Datos, se encuentra en la etapa final de sus estudios. Por un lado, está muy contenta por haber llegado tan lejos, pero por otro, no puede evitar sentirse inquieta. Desde que ingresó a la universidad, una pregunta la ha perseguido: ¿qué tan probable es que pueda ser seleccionada en los lugares donde envíe postulaciones para puestos de trabajo?

Esta duda la mantiene en constante reflexión, especialmente porque sabe que el mercado laboral en Ciencia de Datos es competitivo y exige habilidades no solo técnicas, sino también estratégicas para destacar. Sin embargo, Vale actualmente está completamente enfocada en terminar su tesis de magíster y ha tenido que postergar cualquier preparación específica para enfrentar el desafío de las postulaciones laborales.

Al ver el avance y las habilidades que usted ha demostrado en el curso, Vale decidió proponerle un desafío que le permitirá disminuir la incertidumbre sobre su futuro laboral. Inspirado en sus conocimientos, recolectó un conjunto de datos que contiene información sobre diversos factores que influyen en las decisiones de contratación de empresas al seleccionar entre sus postulantes. Este set de datos incluye los siguientes atributos:

- Age: Edad del candidato
- Gender: Genero del candidato. Male (0), Female (1).
- EducationLevel: Mayor nivel educacional alcanzado por el candidato. Licenciatura Tipo 1 (1), Licenciatura Tipo 2 (2), Maestría (3), PhD. (4).
- ExperienceYears: Años de experiencia profesional.
- PreviousCompanies: Numero de compañías donde el candidato ha trabajado anteriormente.
- DistanceFromCompany: Distancia en kilometros entre la residencia del candidato y la compañía donde postula.
- InterviewScore: Puntaje obtenido en la entrevista por el candidato entre 0 a 100.
- SkillScore: Puntaje obtenido en evaluación de habilidades técnicas por el candidato, entre 0 a 100.
- PersonalityScore: Puntaje obtenido en pruebas de personalidad del candidato, entre 0 a 100.
- RecruitmentStrategy: Estrategia del equipo de reclutamiento. Agresiva (1), Moderada (2), Conservadora (3).

Variable a predecir:
- HiringDecision: Resultado de la postulación. No contratado (0), Contratado (1).

Su objetivo será ayudar a Vale a desarrollar un modelo que le permita predecir, basado en estos factores, si un postulante será contratado o no. Esta herramienta no solo le dará a Vale mayor claridad sobre el impacto de ciertos atributos en la decisión final de contratación, sino que también le permitirá aplicar sus conocimientos de Ciencia de Datos para resolver una pregunta que a muchos estudiantes como a ella les inquieta.

Como estudiante del curso Laboratorio de Programación Científica para Ciencia de Datos, deberá demostrar sus capacidades para preprocesar, analizar y modelar datos, brindándole a Vale una solución robusta y bien fundamentada para su problemática.

`Nota:` El siguiente [enlace](https://www.kaggle.com/datasets/rabieelkharoua/predicting-hiring-decisions-in-recruitment-data/data) contiene el set de datos original.

# **1. Pipeline de Predicción Lineal** (30 Puntos)

<p align="center">
  <img src="https://media.licdn.com/dms/image/v2/D4E22AQHZplrdPyKnvA/feedshare-shrink_2048_1536/feedshare-shrink_2048_1536/0/1713736729086?e=2147483647&v=beta&t=Tad2ulaWkhhDrPRN0PCdXrfuza60PjoJqgLborDyLao" width="500">
</p>

En esta sección buscaremos desplegar un producto utilizando un modelo de clasificación `Random Forest` para determinar **si una persona será contratada o no en un proceso de selección**. Para ello, comenzaremos preparando un pipeline lineal mediante `Airflow`.

## **1.1 Preparando el Pipeline** (15 puntos)

**Primero, asegúrese de tener creada las carpetas `dags`, `plugins` y `logs`**.

Comenzamos preparando un archivo llamado `hiring_functions.py`, el cual guardará en la carpeta `dags` y debe contener lo siguiente:

1. (3 puntos) Una función llamada `create_folders()` que cree una carpeta, la cual utilice la fecha de ejecución como nombre. Adicionalmente, dentro de esta carpeta debe crear las siguientes subcarpetas:
  - raw
  - splits
  - models

  `Hint`: Puede hacer uso de kwargs para obtener la fecha de ejecución mediante el DAG. El siguiente [Enlace](https://airflow.apache.org/docs/apache-airflow/stable/templates-ref.html) le puede ser útil.

2. (3 puntos) Una función llamada `split_data()` que lea el archivo `data_1.csv` de la carepta `raw` y a partir de este, aplique un *hold out*, generando un dataset de entrenamiento y uno de prueba. Luego debe guardar estos nuevos conjuntos de datos en la carpeta `splits`. `Nota:` Utilice un 20% para el conjunto de prueba, mantenga la proporción original en la variable objetivo y fije una semilla.

3. (8 puntos) Cree una función llamada `preprocess_and_train()` que:
  - Lea los set de entrenamiento y prueba de la carpeta `splits`.
  - Cree y aplique un `Pipeline` con una etapa de preprocesamiento. Utilice `ColumnTransformers` para aplicar las transformaciones que estime convenientes. Puede apoyarse del archivo `data_1_report.html` para justificar cualquier paso del preprocesamiento.
  
  - Añada una etapa de entrenamiento utilizando el modelo `RandomForest`.
  
  Esta función **debe crear un archivo `joblib` (análogo a `pickle`) con el pipeline entrenado** en la carepta `models`, además debe **imprimir** el accuracy en el conjunto de prueba y el f1-score de la clase positiva (contratado).
3. (1 punto) Incorpore la función `gradio_interface` en su script, modificando la ruta de acceso a su modelo, de forma que pueda leerlo desde la carepta `models`. Puede realizar las modificaciones que estime necesarias.

`NOTA:` Se permite la creación de funciones auxiliares si lo estiman conveniente.

Función de hiring_functions.py

In [ ]:
"""
Funciones auxiliares para el pipeline de Airflow.
Este contiene funciones reutilizables.
"""
from __future__ import annotations
from pathlib import Path
from datetime import datetime
import os
from airflow.models import Variable

import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import accuracy_score, f1_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier


def create_folders(base_dir: str | None = None, experiment_type: str = "linear", **kwargs):
    """Crea carpetas raw, splits, models para la fecha de ejecución"""
    ds = kwargs.get("ds")
    if not ds:
        ds = datetime.now().strftime("%Y-%m-%d")
    
    if base_dir is None:
        base_dir = Variable.get("BASE_DATA_DIR", "/opt/airflow/data")

    run_dir = Path(base_dir) / experiment_type / ds
    subdirs = ["raw", "splits", "models"]
    
    for subdir in subdirs:
        dir_path = run_dir / subdir
        os.makedirs(dir_path, exist_ok=True)
    
    print(f"Carpetas creadas en: {run_dir}")
    
    ti = kwargs.get("ti")
    if ti:
        ti.xcom_push(key="run_dir", value=str(run_dir))


def split_data(
    base_dir: str | None = None,
    experiment_type: str = "linear",
    target_col: str = "HiringDecision",
    test_size: float = 0.2,
    random_state: int = 42,
    **kwargs):
    """Aplica hold-out 80/20 estratificado"""
    
    ds = kwargs.get("ds")
    if not ds:
        ds = datetime.now().strftime("%Y-%m-%d")
    
    if base_dir is None:
        base_dir = Variable.get("BASE_DATA_DIR", "/opt/airflow/data")
    
    run_dir = Path(base_dir) / experiment_type / ds
    raw_path = run_dir / "raw"
    splits_path = run_dir / "splits"
    splits_path.mkdir(parents=True, exist_ok=True)
    
    csv_path = raw_path / "data_1.csv"
    df = pd.read_csv(csv_path)
    
    print(f"📊 Dataset cargado: {df.shape[0]} filas, {df.shape[1]} columnas")
    
    X = df.drop(columns=[target_col])
    y = df[target_col]
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y)
    
    train_out = splits_path / "train.csv"
    test_out = splits_path / "test.csv"
    
    pd.concat([X_train, y_train], axis=1).to_csv(train_out, index=False)
    pd.concat([X_test, y_test], axis=1).to_csv(test_out, index=False)
    
    print(f"Train: {len(X_train)} muestras")
    print(f"Test: {len(X_test)} muestras")
    
    ti = kwargs.get("ti")
    if ti:
        ti.xcom_push(key="train_path", value=str(train_out))
        ti.xcom_push(key="test_path", value=str(test_out))


def preprocess_and_train(
    base_dir: str | None = None,
    experiment_type: str = "linear",
    target_col: str = "HiringDecision",
    positive_label=1,
    rf_params: dict | None = None,
    **kwargs,
):
    """Preprocesa y entrena RandomForest, guarda pipeline en models/"""
    
    ds = kwargs.get("ds") or datetime.now().strftime("%Y-%m-%d")

    if base_dir is None:
        base_dir = Variable.get("BASE_DATA_DIR", default_var="/opt/airflow/data")

    run_dir = Path(base_dir) / experiment_type / ds
    splits_dir = run_dir / "splits"
    models_dir = run_dir / "models"
    models_dir.mkdir(parents=True, exist_ok=True)

    train_path = splits_dir / "train.csv"
    test_path  = splits_dir / "test.csv"
    
    if not train_path.exists() or not test_path.exists():
        raise FileNotFoundError(f"Faltan splits:\n- {train_path}\n- {test_path}")

    train_df = pd.read_csv(train_path)
    test_df  = pd.read_csv(test_path)

    y_train = train_df[target_col]
    X_train = train_df.drop(columns=[target_col])
    y_test  = test_df[target_col]
    X_test  = test_df.drop(columns=[target_col])

    cat_ordinal = [c for c in ["PreviousCompanies", "EducationLevel"] if c in X_train.columns]

    preprocessor = ColumnTransformer(
        transformers=[
            ("ordinal",
             OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
             cat_ordinal),
        ],
        remainder="passthrough"
    )

    rf_params = rf_params or {
        "n_estimators": 300,
        "random_state": 42,
        "n_jobs": -1
    }
    clf = RandomForestClassifier(**rf_params)

    pipe = Pipeline([
        ("preprocesador", preprocessor),
        ("rf", clf)
    ])

    print("🔧 Entrenando RandomForest...")
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, average="binary", pos_label=positive_label)
    
    print(f"[{ds}] Accuracy (test): {acc:.4f}")
    print(f"[{ds}] F1-Score (clase positiva): {f1:.4f}")

    model_path = models_dir / "model_rf_pipeline.joblib"
    joblib.dump(pipe, model_path)
    print(f"Modelo guardado en: {model_path}")

    ti = kwargs.get("ti")
    if ti:
        ti.xcom_push(key="model_path", value=str(model_path))
        ti.xcom_push(key="metrics", value={"accuracy": float(acc), "f1": float(f1)})

Gradio app

In [ ]:
from pathlib import Path
import os
import tempfile
import joblib
import pandas as pd
import gradio as gr
import io

BASE_DATA_DIR = os.getenv("BASE_DATA_DIR", "/opt/airflow/data")
TARGET_COL = "HiringDecision"

# Leer desde la carpeta models, como lo indica la instrucción
def get_latest_model():
    base_path = Path(BASE_DATA_DIR) / "linear"
    run_dirs = sorted([d for d in base_path.iterdir() if d.is_dir() and d.name.count('-') == 2], reverse=True)
    
    for run_dir in run_dirs:
        model_path = run_dir / "models" / "model_rf_pipeline.joblib"
        if model_path.exists():
            print(f"Modelo encontrado: {model_path}")
            return model_path, run_dir.name
    
    raise FileNotFoundError("No modelo encontrado. Ejecuta el DAG primero.")

def predict_from_json(json_file):
    if json_file is None:
        return pd.DataFrame({"Error": ["⚠️ Sube un archivo JSON"]})
    
    try:
        model_path, run_date = get_latest_model()
        pipe = joblib.load(model_path)
        
        # Lee el contenido del archivo subido
        # json_file es un objeto de archivo temporal
        print(f"📂 Archivo recibido: {json_file.name}")
        
        # Leer JSON directamente desde el objeto de archivo
        df = pd.read_json(json_file.name, orient="records")
        print(f"Datos: {df.shape[0]} filas, {df.shape[1]} columnas")
        
        
        if TARGET_COL in df.columns:
            df = df.drop(columns=[TARGET_COL])
        
        # Predicción
        y_hat = pipe.predict(df)
        
        # Probabilidades
        prob_pos = None
        if hasattr(pipe, "predict_proba"):
            probas = pipe.predict_proba(df)
            if probas.shape[1] >= 2:
                prob_pos = probas[:, 1]
        
        # Resultados
        out = df.copy()
        out["Predicción"] = ["✅ Contratado" if p == 1 else "❌ No Contratado" for p in y_hat]
        if prob_pos is not None:
            out["Probabilidad_Contratación"] = [f"{p:.1%}" for p in prob_pos]
        
        print(f"✅ {len(out)} predicciones generadas (modelo: {run_date})")
        return out
    
    except Exception as e:
        import traceback
        error_msg = f"❌ {str(e)}\n\n{traceback.format_exc()}"
        print(error_msg)
        return pd.DataFrame({"Error": [str(e)]})

iface = gr.Interface(
    fn=predict_from_json,
    inputs=gr.File(label="📁 Sube archivo JSON"),
    outputs=gr.Dataframe(label="📊 Predicciones"),
    title="🎓 Predictor de Contratación",
    description="""
    Sube un JSON con datos para predecir si serán contratados.
    """,
    allow_flagging="never"
)

if __name__ == "__main__":
    print("🚀 Gradio iniciando en puerto 7860...")
    print(f"📂 BASE_DATA_DIR: {BASE_DATA_DIR}")
    
    try:
        model_path, run_date = get_latest_model()
        print(f"✅ Modelo cargado: {run_date}")
    except:
        print("⚠️  Esperando modelo...")
    
    iface.launch(server_name="0.0.0.0", server_port=7860, share=True)

## **1.2 Creando Nuestro DAG** (15 puntos)

<p align="center">
  <img src="https://media.tenor.com/a_yibuZQgngAAAAM/elle-woods.gif" width="400">
</p>

Con las funciones del pipeline ya creadas, ahora vamos a proceder a crear un Directed Acyclic Graph (DAG). Para ello, se le pide lo siguiente:

- (10 puntos) Cree un segundo archivo llamado `dag_lineal.py` y guardelo en la carpeta dags. Este script debe seguir la siguiente estructura (Ver imagen de referencia):

    0. Inicialice un DAG con fecha de inicio el 1 de octubre de 2024, ejecución manual y **sin backfill**. Asigne un `dag_id` que pueda reconocer facilmente, como `hiring_lineal`, etc.
    1. Debe comenzar con un marcador de posición que indique el inicio del pipeline.
    2. Cree una carpeta correspondiente a la ejecución del pipeline y cree las subcarpetas `raw`, `splits` y `models` mediante la función `create_folders()`.
    3. Debe descargar el archivo `data_1.csv` del siguiente [enlace](https://gitlab.com/eduardomoyab/laboratorio-13/-/raw/main/files/data_1.csv). Debe guardar el archivo en la carpeta raw de la ejecución correspondiente.`Hint:` Le puede ser útil el comando `curl -o <path de guardado> <enlace con los datos>`.
    4. Debe aplicar un hold out mediante la función `split_data()` de su archivo creado en la subsección anterior.
    5. Debe aplicar el preprocesamiento y el entrenamiento del modelo mediante la función `preprocess_and_train()`.
    6. Finalmente, debe montar una interfaz en gradio donde pueda cargar un archivo ``json``.


- (3 puntos) Cree un `DockerFile` para montar un contenedor que contenga Airflow. Adicionalmente, cree una carpeta llamada dags donde guardará el script.py creado anteriormente.

    `Nota:` Para la imagen, se recomienda utilizar python 3.10-slim. Adicionalmente, puede instalar `curl` mediante la siguiente linea de código: `RUN apt-get update && apt-get install -y curl`.

- Construya el contenedor en Docker y acceda a la aplicación web de Airflow mediante el siguiente [enlace](http://localhost:8080/). Inicie sesión, acceda al DAG creado y ejecute de forma manual su pipeline.

- (2 puntos) Acceda a la URL pública de Gradio e ingrese el archivo `vale_data.json` a su modelo. ¿Que predicción entregó el modelo para Vale? Adjunte imágenes de su resultado. `Hint:` Puede acceder a los `logs` para obtener los prints y la URL pública.

`Hint:` Recuerde que puede entregar `kwargs` a sus funciones, como por ejemplo la fecha de ejecución `ds`.

**Para esta sección, debe adjuntar todos los scripts creados junto a su notebook en la entrega, ya que serán ejecutados para validar el funcionamiento. Para justificar sus respuestas, adicionaslmente puede utilizar imágenes de apoyo, como screenshots.**

DAG de referencia:
<p align="center">
  <img src="https://drive.google.com/uc?id=1iwDgECZfFeWq1dl433tMa6_3CNF9cn1L" width="1200">
</p>




In [ ]:
#Inserte código aqui
from __future__ import annotations
from datetime import datetime

from airflow import DAG
from airflow.operators.empty import EmptyOperator
from airflow.operators.python import PythonOperator
from airflow.operators.bash import BashOperator
from airflow.models import Variable

#from hiring_functions import create_folders, split_data, preprocess_and_train

# Configuración
DEFAULT_BASE_DIR = Variable.get("BASE_DATA_DIR", default_var="/opt/airflow/data")
DATA_URL = "https://gitlab.com/eduardomoyab/laboratorio-13/-/raw/main/files/data_1.csv"

with DAG(
    dag_id="hiring_lineal",
    start_date=datetime(2024, 10, 1),
    schedule=None,
    catchup=False,
    tags=["hiring", "pipeline", "rf"],
) as dag:

    start = EmptyOperator(task_id="start_pipeline")

    make_dirs = PythonOperator(
        task_id="create_folders",
        python_callable=create_folders,
        op_kwargs={"base_dir": DEFAULT_BASE_DIR, "experiment_type": "linear"},
    )

    download_data = BashOperator(
    task_id="download_data",
    bash_command="""
        mkdir -p {{ params.base_dir }}/linear/{{ ds }}/raw &&
        curl -L -f -o {{ params.base_dir }}/linear/{{ ds }}/raw/data_1.csv {{ params.url }} &&
        echo 'Archivo descargado exitosamente' &&
        ls -lh {{ params.base_dir }}/linear/{{ ds }}/raw/data_1.csv
    """,
    params={
        "base_dir": DEFAULT_BASE_DIR,
        "url": DATA_URL,
    },
    )
    split = PythonOperator(
        task_id="split_data",
        python_callable=split_data,
        op_kwargs={
            "base_dir": DEFAULT_BASE_DIR,
            "experiment_type": "linear",
            "target_col": "HiringDecision",
            "test_size": 0.2,
            "random_state": 42,
        },
    )

    train = PythonOperator(
        task_id="preprocess_and_train",
        python_callable=preprocess_and_train,
        op_kwargs={
            "base_dir": DEFAULT_BASE_DIR,
            "experiment_type": "linear",
            "target_col": "HiringDecision",
            "positive_label": 1,
        },
    )

    end = EmptyOperator(task_id="pipeline_completed")

    start >> make_dirs >> download_data >> split >> train >> end

Dockerfile.airflow

`FROM apache/airflow:2.7.3-python3.10`

`USER root`
`RUN apt-get update && apt-get install -y curl && rm -rf /var/lib/apt/lists/*`

`USER airflow`
`COPY requirements-airflow.txt /tmp/`
`RUN pip install --no-cache-dir -r /tmp/requirements-airflow.txt`

Dockerfile.gradio

`FROM python:3.10-slim`

`WORKDIR /app`

`COPY requirements-gradio.txt .`
`RUN pip install --no-cache-dir -r requirements-gradio.txt`

`CMD ["python", "/app/gradio_app.py"]`

docker-compose.yml

```yaml
version: '3.8'

services:
  airflow:
    build:
      context: .
      dockerfile: Dockerfile.airflow
    container_name: airflow-lab9
    ports:
      - "8080:8080"
    volumes:
      - ./dags:/opt/airflow/dags
      - ./logs:/opt/airflow/logs
      - ./plugins:/opt/airflow/plugins
      - ./data:/opt/airflow/data
    environment:
      - AIRFLOW__CORE__EXECUTOR=SequentialExecutor
      - AIRFLOW__CORE__LOAD_EXAMPLES=False
      - AIRFLOW__DATABASE__SQL_ALCHEMY_CONN=sqlite:////opt/airflow/airflow.db
      - BASE_DATA_DIR=/opt/airflow/data
      - _AIRFLOW_DB_MIGRATE=true
      - _AIRFLOW_WWW_USER_CREATE=true
      - _AIRFLOW_WWW_USER_USERNAME=admin
      - _AIRFLOW_WWW_USER_PASSWORD=admin
    command: >
      bash -c "airflow db migrate &&
               airflow users create --username admin --password admin --firstname Admin --lastname User --role Admin --email admin@example.com || true &&
               airflow webserver & airflow scheduler"
    restart: unless-stopped

  gradio:
    build:
      context: .
      dockerfile: Dockerfile.gradio
    container_name: gradio-lab9
    ports:
      - "7860:7860"
    volumes:
      - ./data:/opt/airflow/data
      - ./gradio_app.py:/app/gradio_app.py
    environment:
      - BASE_DATA_DIR=/opt/airflow/data
    restart: unless-stopped
    depends_on:
      - airflow
```

Entonces, en el directorio raíz se deja Dockerfile.airflow, Dockerfile.gradio y docker-compose.yml. Se tienen tres carpetas: dags, plugins y logs. En la carpeta dags se dejan los archivos dag_lineal.py y hiring_functions.py.

Luego, se corre el docker-compose, dando lugar a : 

(ver https://github.com/danmc899/MDS7202/blob/main/Entregables_Lab9/dags/Images/docker.png si no carga la imagen)
<p align="center"><img src="dags/Images/docker.png" width="1200"></p>


A continuación, se accede a la aplicación web de Airflow mediante el siguiente enlace: http://localhost:8080/. Se inicia sesión con las credenciales admin/admin, se activa el dag creado y se ejecuta manualmente : 

(ver https://github.com/danmc899/MDS7202/blob/main/Entregables_Lab9/dags/Images/Ejecuci%C3%B3n%20lineal.png si no carga la imagen)
<p align="center"><img src="dags/Images/Ejecución lineal.png" width="1200"></p>

Entonces, en el directorio de data/linear/<fecha_de_ejecución>/models/ se guarda el modelo entrenado model_rf_pipeline.joblib. A continuación, se muestra el flujo de trabajo en Airflow : 

(ver https://github.com/danmc899/MDS7202/blob/main/Entregables_Lab9/dags/Images/Flujo%20lineal.png si no carga la imagen)
<p align="center"><img src="dags/Images/Flujo lineal.png" width="1200"></p>

Por último, se accede a la URL pública de Gradio con link http://localhost:7860/ y se ingresa el archivo vale_data.json al modelo. El resultado es : 

(ver https://github.com/danmc899/MDS7202/blob/main/Entregables_Lab9/dags/Images/Gradio%20lineal.png si no carga la imagen)
<p align="center"><img src="dags/Images/Gradio lineal.png" width="1200"></p>



# **2. Paralelizando el Pipeline** (30 puntos)

<p align="center">
  <img src="https://i.gifer.com/8LNL.gif" width="400">
</p>



Al ver los resultados obtenidos, Vale queda muy contenta con el clasificador. Sin embargo, le aparecen algunas dudas respecto al funcionamiento del pipeline. Primero le comenta que es posible que en un futuro tenga nuevos datos que podrían ser útiles para realizar nuevos entrenamientos, por lo que sería ideal si este pipeline se fuera ejecutando de forma periódica y **NO** de forma manual. Además, Vale le menciona que le gustaría explorar el desempeño de otros modelos además de `Random Forest`, de forma que el pipeline seleccione de forma automática el modelo con mejor desempeño para luego hacer la predicción de Vale.

## **2.1 Preparando un Nuevo Pipeline** (15 puntos)

<p align="center">
  <img src="https://media.tenor.com/gnA7-5TewXMAAAAM/elle-woods.gif" width="400">
</p>

De acuerdo a lo que le comentó Vale, usted decide crear un nuevo script con las funciones que utilizará su pipeline. Por ende, dentro de la carpeta `dags`, usted creará el archivo `hiring_dynamic_functions.py` el cual debe contener:

1. (2 puntos) Una función llamada `create_folders()` que cree una carpeta, la cual utilice la fecha de ejecución como nombre. Adicionalmente, dentro de esta carpeta debe crear las siguientes subcarpetas:
  - raw
  - preprocessed
  - splits
  - models
2. (2 puntos) Una función llamada `load_ands_merge()` que lea desde la carpeta `raw` los archivos `data_1.csv`y `data_2.csv` en caso de estar disponible. Luego concatene estos y genere un nuevo archivo resultante, guardándolo en la carpeta `preprocessed`.

3. (2 puntos) Una función llamada `split_data()` que lea la data guardada en la carpeta `preprocessed` y realice un hold out sobre esta data. Esta función debe crear un conjunto de entrenamiento y uno de prueba. Mantenga una semilla y 20% para el conjunto de prueba. Guarde los conjuntos resultantes en la carpeta `splits`.

4. (6 puntos) Una función llamada `train_model()` que reciba un modelo de clasificación.
    - La función debe comenzar leyendo el conjunto de entrenamiento desde la carpeta `spits`.
    - Esta debe crear y aplicar un `Pipeline` con una etapa de preprocesamiento. Utilice `ColumnTransformers` para aplicar las transformaciones que estime convenientes.
    - Añada una etapa de entrenamiento utilizando un modelo que ingrese a la función.
  
  Esta función **debe crear un archivo joblib con el pipeline entrenado**. Guarde el modelo con un nombre que le permita una facil identificación dentro de la carpeta `models`.

5. (3 puntos) Una función llamada `evaluate_models()` que reciba sus modelos entrenados desde la carpeta `models`, evalúe su desempeño mediante `accuracy` en el conjunto de prueba y seleccione el mejor modelo obtenido. Luego guarde el mejor modelo como archivo `.joblib`. Su función debe imprimir el nombre del modelo seleccionado y el accuracy obtenido.

In [ ]:
#Inserte código aqui
from __future__ import annotations
from pathlib import Path
from datetime import datetime
import os

from airflow.models import Variable
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
from glob import glob


def create_folders(base_dir: str | None = None, experiment_type: str = "dynamic", **kwargs):
    """
    Crea una carpeta con nombre igual a la fecha de ejecución (ds) y,
    dentro de ella, crea las carpetas raw, preprocessed, splits y models.
    
    Params
    ------
    base_dir: str | None
        Directorio base donde crear la estructura. Si es None, intenta
        leer Variable ("BASE_DATA_DIR"). Si no, usa "/opt/airflow/data".
    experiment_type: str
        Tipo de experimento: "linear" o "dynamic"
    """
    
    # Obtener la fecha de ejecución
    ds = kwargs.get("ds")
    if not ds:
        ds = datetime.now().strftime("%Y-%m-%d")
    
    # Resolver el directorio base
    if base_dir is None:
        base_dir = Variable.get("BASE_DATA_DIR", "/opt/airflow/data")

    # Construir paths con experiment_type
    run_dir = Path(base_dir) / experiment_type / ds
    subdirs = ["raw", "preprocessed", "splits", "models"]
    
    # Crear carpetas
    for subdir in subdirs:
        dir_path = run_dir / subdir
        os.makedirs(dir_path, exist_ok=True)
    
    print(f"Carpetas creadas en: {run_dir}")
    
    # Publicar por XCom
    if kwargs.get("ti"):
        kwargs["ti"].xcom_push(key="run_dir", value=str(run_dir))


def load_and_merge(base_dir: str | None = None, experiment_type: str = "dynamic", **kwargs):
    """
    Lee data_1.csv y data_2.csv (si existe) desde la carpeta raw,
    los concatena y guarda el resultado en preprocessed/merged_data.csv.
    
    Params
    ------
    base_dir: str | None
        Directorio base que contiene la carpeta de la corrida (YYYY-MM-DD).
    experiment_type: str
        Tipo de experimento: "linear" o "dynamic"
    """
    
    # Obtener fecha de ejecución
    ds = kwargs.get("ds")
    if not ds:
        ds = datetime.now().strftime("%Y-%m-%d")
    
    # Resolver el directorio base
    if base_dir is None:
        base_dir = Variable.get("BASE_DATA_DIR", "/opt/airflow/data")
    
    run_dir = Path(base_dir) / experiment_type / ds
    raw_path = run_dir / "raw"
    preprocessed_path = run_dir / "preprocessed"
    preprocessed_path.mkdir(parents=True, exist_ok=True)
    
    # Buscar archivos disponibles
    data_1_path = raw_path / "data_1.csv"
    data_2_path = raw_path / "data_2.csv"
    
    dfs = []
    
    if data_1_path.exists():
        print(f"Leyendo {data_1_path}")
        dfs.append(pd.read_csv(data_1_path))
    
    if data_2_path.exists():
        print(f"Leyendo {data_2_path}")
        dfs.append(pd.read_csv(data_2_path))
    
    if not dfs:
        raise FileNotFoundError(f"No se encontraron archivos en {raw_path}")
    
    # Concatenar datasets
    merged_df = pd.concat(dfs, axis=0, ignore_index=True)
    
    # Guardar en preprocessed
    output_path = preprocessed_path / "merged_data.csv"
    merged_df.to_csv(output_path, index=False)
    
    print(f"Datos concatenados guardados en: {output_path}")
    print(f"Forma del dataset: {merged_df.shape}")
    
    # Publicar por XCom
    if kwargs.get("ti"):
        kwargs["ti"].xcom_push(key="merged_path", value=str(output_path))


def split_data(
    base_dir: str | None = None,
    experiment_type: str = "dynamic",
    target_col: str = "HiringDecision",
    test_size: float = 0.2,
    random_state: int = 42,
    **kwargs
):
    """
    Lee merged_data.csv desde preprocessed y aplica un hold-out 80/20,
    estratificado por la variable objetivo. Guarda train/test en splits/.
    
    Params
    ------
    base_dir : str | None
        Directorio base que contiene la carpeta de la corrida.
    experiment_type: str
        Tipo de experimento: "linear" o "dynamic"
    target_col : str
        Nombre de la variable objetivo para estratificar.
    test_size : float
        Proporción de test.
    random_state : int
        Semilla para replicabilidad.
    """
    
    # Obtener fecha de ejecución
    ds = kwargs.get("ds")
    if not ds:
        ds = datetime.now().strftime("%Y-%m-%d")
    
    # Resolver el directorio base
    if base_dir is None:
        base_dir = Variable.get("BASE_DATA_DIR", "/opt/airflow/data")
    
    run_dir = Path(base_dir) / experiment_type / ds
    preprocessed_path = run_dir / "preprocessed"
    splits_path = run_dir / "splits"
    splits_path.mkdir(parents=True, exist_ok=True)
    
    # Cargar datos procesados
    merged_path = preprocessed_path / "merged_data.csv"
    if not merged_path.exists():
        raise FileNotFoundError(f"No se encontró {merged_path}")
    
    df = pd.read_csv(merged_path)
    
    # Separar características y objetivo
    X = df.drop(columns=[target_col])
    y = df[target_col]
    
    # Hold out
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, 
        test_size=test_size, 
        random_state=random_state, 
        stratify=y
    )
    
    # Guardar los datasets
    train_out = splits_path / "train.csv"
    test_out = splits_path / "test.csv"
    
    pd.concat([X_train, y_train], axis=1).to_csv(train_out, index=False)
    pd.concat([X_test, y_test], axis=1).to_csv(test_out, index=False)
    
    print(f"Train guardado en: {train_out} - Shape: {X_train.shape}")
    print(f"Test guardado en: {test_out} - Shape: {X_test.shape}")
    
    # Publicar por XCom
    if kwargs.get("ti"):
        kwargs["ti"].xcom_push(key="train_path", value=str(train_out))
        kwargs["ti"].xcom_push(key="test_path", value=str(test_out))


def train_model(
    model,
    model_name: str,
    base_dir: str | None = None,
    experiment_type: str = "dynamic",
    target_col: str = "HiringDecision",
    **kwargs
):
    """
    Entrena un modelo de clasificación con preprocesamiento y lo guarda.
    
    Params
    ------
    model : estimator
        Modelo de sklearn a entrenar.
    model_name : str
        Nombre identificador del modelo para guardar el archivo.
    base_dir : str | None
        Directorio base de la corrida.
    experiment_type: str
        Tipo de experimento: "linear" o "dynamic"
    target_col : str
        Nombre de la columna objetivo.
    """
    
    # Obtener fecha de ejecución
    ds = kwargs.get("ds") or datetime.now().strftime("%Y-%m-%d")
    
    # Resolver el directorio base
    if base_dir is None:
        base_dir = Variable.get("BASE_DATA_DIR", default_var="/opt/airflow/data")
    
    run_dir = Path(base_dir) / experiment_type / ds
    splits_dir = run_dir / "splits"
    models_dir = run_dir / "models"
    models_dir.mkdir(parents=True, exist_ok=True)
    
    # Cargar datos de entrenamiento
    train_path = splits_dir / "train.csv"
    if not train_path.exists():
        raise FileNotFoundError(f"No se encontró {train_path}")
    
    train_df = pd.read_csv(train_path)
    
    # Separar características y objetivo
    y_train = train_df[target_col]
    X_train = train_df.drop(columns=[target_col])
    
    # Preprocesamiento
    cat_ordinal = [c for c in ["PreviousCompanies", "EducationLevel"] 
                   if c in X_train.columns]
    
    preprocessor = ColumnTransformer(
        transformers=[
            ("ordinal",
             OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
             cat_ordinal),
        ],
        remainder="passthrough"
    )
    
    # Pipeline completo
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    
    # Entrenar
    print(f"Entrenando modelo: {model_name}")
    pipe.fit(X_train, y_train)
    
    # Guardar modelo
    model_path = models_dir / f"{model_name}.joblib"
    joblib.dump(pipe, model_path)
    
    print(f"Modelo {model_name} guardado en: {model_path}")
    
    # Publicar por XCom
    if kwargs.get("ti"):
        kwargs["ti"].xcom_push(key=f"model_{model_name}", value=str(model_path))


def evaluate_models(
    base_dir: str | None = None,
    experiment_type: str = "dynamic",
    target_col: str = "HiringDecision",
    **kwargs
):
    """
    Evalúa todos los modelos guardados en la carpeta models/ usando el conjunto
    de prueba, selecciona el mejor por accuracy y lo guarda como best_model.joblib.
    
    Params
    ------
    base_dir : str | None
        Directorio base de la corrida.
    experiment_type: str
        Tipo de experimento: "linear" o "dynamic"
    target_col : str
        Nombre de la columna objetivo.
    """
    
    # Obtener fecha de ejecución
    ds = kwargs.get("ds") or datetime.now().strftime("%Y-%m-%d")
    
    # Resolver el directorio base
    if base_dir is None:
        base_dir = Variable.get("BASE_DATA_DIR", default_var="/opt/airflow/data")
    
    run_dir = Path(base_dir) / experiment_type / ds
    splits_dir = run_dir / "splits"
    models_dir = run_dir / "models"
    
    # Cargar datos de prueba
    test_path = splits_dir / "test.csv"
    if not test_path.exists():
        raise FileNotFoundError(f"No se encontró {test_path}")
    
    test_df = pd.read_csv(test_path)
    y_test = test_df[target_col]
    X_test = test_df.drop(columns=[target_col])
    
    # Buscar todos los modelos
    model_files = list(models_dir.glob("*.joblib"))
    # Excluir best_model si ya existe
    model_files = [f for f in model_files if f.stem != "best_model"]
    
    if not model_files:
        raise FileNotFoundError(f"No se encontraron modelos en {models_dir}")
    
    print(f"Evaluando {len(model_files)} modelos en experimento '{experiment_type}'...")
    
    results = []
    
    for model_path in model_files:
        model_name = model_path.stem
        pipe = joblib.load(model_path)
        
        # Predecir
        y_pred = pipe.predict(X_test)
        
        # Calcular accuracy
        acc = accuracy_score(y_test, y_pred)
        
        results.append({
            "model_name": model_name,
            "accuracy": acc,
            "model_path": model_path
        })
        
        print(f"  {model_name}: Accuracy = {acc:.4f}")
    
    # Seleccionar el mejor modelo
    best_result = max(results, key=lambda x: x["accuracy"])
    
    print(f"\n Mejor modelo ({experiment_type}): {best_result['model_name']}")
    print(f"Accuracy: {best_result['accuracy']:.4f}")
    
    # Guardar el mejor modelo como best_model.joblib
    best_model_path = models_dir / "best_model.joblib"
    best_pipe = joblib.load(best_result["model_path"])
    joblib.dump(best_pipe, best_model_path)

    print(f"Guardado en: {best_model_path}")

    # Publicar por XCom
    if kwargs.get("ti"):
        kwargs["ti"].xcom_push(key="best_model_name", value=best_result["model_name"])
        kwargs["ti"].xcom_push(key="best_model_accuracy", value=best_result["accuracy"])
        kwargs["ti"].xcom_push(key="best_model_path", value=str(best_model_path))

## **2.2 Componiendo un nuevo DAG** (15 puntos)

<p align="center">
  <img src="https://67.media.tumblr.com/bfa5208006dc3f404ec08e8c3195cf2c/tumblr_obg9tgnLfX1u9e9f2o2_r1_500.gif" width="500">
</p>

Con las nuevas funciones, se debe crear el nuevo nuevo DAG. Para ello, cree un nuevo script en la carpeta `dags`, llamandolo `dag_dynamic.py`. Este script debe contener la siguiente estructura:

1. (1 punto) Inicialice un DAG con fecha de inicio el 1 de octubre de 2024, el cual se debe ejecutar el día 5 de cada mes a las 15:00 UTC. Utilice un `dag_id` interpretable para identificar fácilmente. **Habilite el backfill** para que pueda ejecutar tareas programadas desde fechas pasadas.
2. (1 punto) Comience con un marcador de posición que indique el inicio del pipeline.
3. (2 puntos) Cree una carpeta correspondiente a la ejecución del pipeline y cree las subcarpetas `raw`, `preprocessed`, `splits` y `models` mediante la función `create_folders()`.
4. (2 puntos) Implemente un `Branching`que siga la siguiente lógica:
  - Fechas previas al 1 de noviembre de 2024: Se descarga solo `data_1.csv`
  - Desde el 1 de noviembre del 2024: descarga `data_1.csv` y `data_2.csv`.
  En el siguiente [enlace](https://gitlab.com/eduardomoyab/laboratorio-13/-/raw/main/files/data_2.csv) puede descargar `data_2.csv`.
5. (1 punto) Cree una tarea que concatene los datasets disponibles mediante la función `load_and_merge()`. Configure un `Trigger` para que la tarea se ejecute si encuentra disponible **como mínimo** uno de los archivos.
6. (1 punto) Aplique el hold out al dataset mediante la función `split_data()`, obteniendo un conjunto de entrenamiento y uno de prueba.
7. (2 puntos) Realice 3 entrenamientos en paralelo:
  - Un modelo Random Forest.
  - 2 modelos a elección.
  Asegúrese de guardar sus modelos entrenados con nombres distintivos. Utilice su función `train_model()` para ello.
8. (2 puntos) Mediante la función `evaluate_models()`, evalúe los modelos entrenados, registrando el accuracy de cada modelo en el set de prueba. Luego debe imprimir el mejor modelo seleccionado y su respectiva métrica. Configure un `Trigger` para que la tarea se ejecute solamente si los 3 modelos fueron entrenados y guardados.

`Hint:` Recuerde que puede entregar `kwargs` a sus funciones, como por ejemplo la fecha de ejecución `ds`.

Una vez creado el script, vuelva a construir el contenedor en Docker, acceda a la aplicación web de Airflow, ejecute su pipeline y muestre sus resultados. Adjunte imágenes que ayuden a mostrar el proceso y sus resultados.

Adicionalmente, responda (1 c/u):

- ¿Cual es el accuracy de cada modelo en la ejecución de octubre? ¿Se obtienen los mismos resultados a partir de Noviembre?
- Analice como afectó el añadir datos a sus modelos mediante el desempeño del modelo y en costo computacional.
- Muestre el esquema de su DAG ejecutado en octubre y en noviembre.


`Nota:` Para esta sección no debe implementar la tarea en gradio, solamente se espera determinar el mejor modelo y comparar el desempeño obtenido.

**IMPORTANTE: Para esta sección, debe adjuntar todos los scripts creados junto a su notebook en la entrega, ya que serán ejecutados para validar el funcionamiento. Para justificar sus respuestas, adicionaslmente puede utilizar imágenes de apoyo, como screenshots.**

In [ ]:
#Inserte código aqui
from __future__ import annotations
from datetime import datetime
from pathlib import Path

from airflow import DAG
from airflow.operators.empty import EmptyOperator
from airflow.operators.python import PythonOperator, BranchPythonOperator
from airflow.operators.bash import BashOperator
from airflow.models import Variable
from airflow.utils.trigger_rule import TriggerRule

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

# Importar funciones del archivo de funciones dinámicas
#from hiring_dynamic_functions import (
#    create_folders,
#    load_and_merge,
#    split_data,
#    train_model,
#    evaluate_models
#)

# Configuración
DEFAULT_BASE_DIR = Variable.get("BASE_DATA_DIR", default_var="/opt/airflow/data")
DATA_1_URL = "https://gitlab.com/eduardomoyab/laboratorio-13/-/raw/main/files/data_1.csv"
DATA_2_URL = "https://gitlab.com/eduardomoyab/laboratorio-13/-/raw/main/files/data_2.csv"


# Función para decidir qué datos descargar según la fecha
def decide_download_branch(**kwargs):
    """
    Decide qué rama tomar según la fecha de ejecución:
    - Antes del 1 de noviembre 2024: solo data_1
    - Desde el 1 de noviembre 2024: data_1 y data_2
    """
    ds = kwargs.get("ds")
    execution_date = datetime.strptime(ds, "%Y-%m-%d")
    cutoff_date = datetime(2024, 11, 1)
    
    if execution_date < cutoff_date:
        print(f"Fecha {ds} < 2024-11-01: descargando solo data_1")
        return "download_data_1_only"
    else:
        print(f"Fecha {ds} >= 2024-11-01: descargando data_1 y data_2")
        return ["download_data_1", "download_data_2"]

# Definición del DAG
with DAG(
    dag_id="hiring_dynamic",
    start_date=datetime(2024, 10, 1),
    schedule="0 15 5 * *",  # Día 5 de cada mes a las 15:00 UTC
    catchup=True,  # Habilitar backfill
    tags=["hiring", "dynamic", "parallel"],
) as dag:

    # Marcador de inicio
    start = EmptyOperator(task_id="start_pipeline")

    # Crear carpetas
    make_dirs = PythonOperator(
        task_id="create_folders",
        python_callable=create_folders,
        op_kwargs={"base_dir": DEFAULT_BASE_DIR, "experiment_type": "dynamic"},
    )

    # Branch: decidir qué datos descargar
    branch_download = BranchPythonOperator(
        task_id="decide_download",
        python_callable=decide_download_branch,
    )

    # a. Descargar solo data_1
    download_data_1_only = BashOperator(
    task_id="download_data_1_only",
    bash_command="""
        RAW_DIR="{{ params.base_dir }}/dynamic/{{ ds }}/raw"
        mkdir -p "$RAW_DIR"
        curl -L -f -o "$RAW_DIR/data_1.csv" {{ params.url }}
        echo 'Archivo data_1.csv descargado exitosamente'
        ls -lh "$RAW_DIR/data_1.csv"
    """,
    params={
        "base_dir": DEFAULT_BASE_DIR,
        "url": DATA_1_URL,
    },
    )

    # b. Descargar data_1 y data_2
    download_data_1 = BashOperator(
    task_id="download_data_1",
    bash_command="""
        RAW_DIR="{{ params.base_dir }}/dynamic/{{ ds }}/raw"
        mkdir -p "$RAW_DIR"
        curl -L -f -o "$RAW_DIR/data_1.csv" {{ params.url }}
        echo 'Archivo data_1.csv descargado exitosamente'
        ls -lh "$RAW_DIR/data_1.csv"
    """,
    params={
        "base_dir": DEFAULT_BASE_DIR,
        "url": DATA_1_URL,
    },
    )

    download_data_2 = BashOperator(
    task_id="download_data_2",
    bash_command="""
        RAW_DIR="{{ params.base_dir }}/dynamic/{{ ds }}/raw"
        mkdir -p "$RAW_DIR"
        curl -L -f -o "$RAW_DIR/data_2.csv" {{ params.url }}
        echo 'Archivo data_2.csv descargado exitosamente'
        ls -lh "$RAW_DIR/data_2.csv"
    """,
    params={
        "base_dir": DEFAULT_BASE_DIR,
        "url": DATA_2_URL,
    },
    )

    # Tarea dummy para unir las ramas
    download_complete = EmptyOperator(
        task_id="download_complete",
        trigger_rule=TriggerRule.NONE_FAILED_MIN_ONE_SUCCESS,
    )

    # Concatenar datasets disponibles
    merge_data = PythonOperator(
        task_id="load_and_merge",
        python_callable=load_and_merge,
        op_kwargs={"base_dir": DEFAULT_BASE_DIR, "experiment_type": "dynamic"},
        trigger_rule=TriggerRule.NONE_FAILED_MIN_ONE_SUCCESS,
    )

    # Split train/test
    split = PythonOperator(
        task_id="split_data",
        python_callable=split_data,
        op_kwargs={
            "base_dir": DEFAULT_BASE_DIR,
            "experiment_type": "dynamic",
            "target_col": "HiringDecision",
            "test_size": 0.2,
            "random_state": 42,
        },
    )

    # Entrenamientos en paralelo
    # Modelo 1: Random Forest
    train_rf = PythonOperator(
        task_id="train_random_forest",
        python_callable=train_model,
        op_kwargs={
            "model": RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
            "model_name": "random_forest",
            "base_dir": DEFAULT_BASE_DIR,
            "experiment_type": "dynamic",
            "target_col": "HiringDecision",
        },
    )

    # Modelo 2: Gradient Boosting
    train_gb = PythonOperator(
        task_id="train_gradient_boosting",
        python_callable=train_model,
        op_kwargs={
            "model": GradientBoostingClassifier(n_estimators=200, random_state=42),
            "model_name": "gradient_boosting",
            "base_dir": DEFAULT_BASE_DIR,
            "experiment_type": "dynamic",
            "target_col": "HiringDecision",
        },
    )

    # Modelo 3: Logistic Regression
    train_lr = PythonOperator(
        task_id="train_logistic_regression",
        python_callable=train_model,
        op_kwargs={
            "model": LogisticRegression(max_iter=1000, random_state=42),
            "model_name": "logistic_regression",
            "base_dir": DEFAULT_BASE_DIR,
            "experiment_type": "dynamic",
            "target_col": "HiringDecision",
        },
    )

    # Evaluación y selección del mejor modelo
    evaluate = PythonOperator(
        task_id="evaluate_models",
        python_callable=evaluate_models,
        op_kwargs={
            "base_dir": DEFAULT_BASE_DIR,
            "experiment_type": "dynamic",
            "target_col": "HiringDecision",
        },
        trigger_rule=TriggerRule.ALL_SUCCESS,  # Solo si todos los modelos se entrenaron
    )

    # Marcador de fin
    end = EmptyOperator(task_id="end_pipeline")

    # Definir dependencias
    start >> make_dirs >> branch_download

    # Rama 1: Solo data_1
    branch_download >> download_data_1_only >> download_complete

    # Rama 2: data_1 y data_2
    branch_download >> download_data_1 >> download_complete
    branch_download >> download_data_2 >> download_complete

    # Flujo principal
    download_complete >> merge_data >> split

    # Entrenamientos en paralelo
    split >> [train_rf, train_gb, train_lr]

    # Evaluación después de todos los entrenamientos
    [train_rf, train_gb, train_lr] >> evaluate >> end

Entonces, nuevamente se ejecuta el docker-compose, y se accede a la aplicación web de Airflow mediante el siguiente enlace: http://localhost:8080/. Se inicia sesión con las credenciales admin/admin, se activa el dag creado y se programa su ejecución: 

(ver https://github.com/danmc899/MDS7202/blob/main/Entregables_Lab9/dags/Images/Ejecuci%C3%B3n%20dynamic.png si no carga la imagen)
<p align="center"><img src="dags/Images/Ejecución dynamic.png" width="1200"></p>

Luego, se obtiene este directorio donde se guardan los resultados: 

(ver https://github.com/danmc899/MDS7202/blob/main/Entregables_Lab9/dags/Images/Resultados%20dynamic.png si no carga la imagen)
<p align="center"><img src="dags/Images/Resultados dynamic.png" width="300"></p>

A continuación, se muestra el flujo de trabajo en Airflow: 

(ver https://github.com/danmc899/MDS7202/blob/main/Entregables_Lab9/dags/Images/Flujo%20dynamic.png si no carga la imagen)
<p align="center"><img src="dags/Images/Flujo dynamic.png" width="1200"></p>

Con respecto a los resultados, se accede a los logs de la ejecución de octubre y noviembre para comparar los resultados. Para octubre:

```yaml
*** Found local files:
***   * /opt/airflow/logs/dag_id=hiring_dynamic/run_id=scheduled__2024-10-05T15:00:00+00:00/task_id=evaluate_models/attempt=1.log
[2025-10-28, 00:41:56 UTC] {taskinstance.py:1159} INFO - Dependencies all met for dep_context=non-requeueable deps ti=<TaskInstance: hiring_dynamic.evaluate_models scheduled__2024-10-05T15:00:00+00:00 [queued]>
[2025-10-28, 00:41:56 UTC] {taskinstance.py:1159} INFO - Dependencies all met for dep_context=requeueable deps ti=<TaskInstance: hiring_dynamic.evaluate_models scheduled__2024-10-05T15:00:00+00:00 [queued]>
[2025-10-28, 00:41:56 UTC] {taskinstance.py:1361} INFO - Starting attempt 1 of 1
[2025-10-28, 00:41:56 UTC] {taskinstance.py:1382} INFO - Executing <Task(PythonOperator): evaluate_models> on 2024-10-05 15:00:00+00:00
[2025-10-28, 00:41:56 UTC] {standard_task_runner.py:57} INFO - Started process 1133 to run task
[2025-10-28, 00:41:56 UTC] {standard_task_runner.py:84} INFO - Running: ['airflow', 'tasks', 'run', 'hiring_dynamic', 'evaluate_models', 'scheduled__2024-10-05T15:00:00+00:00', '--job-id', '50', '--raw', '--subdir', 'DAGS_FOLDER/dag_dynamic.py', '--cfg-path', '/tmp/tmpq_k6afqu']
[2025-10-28, 00:41:56 UTC] {standard_task_runner.py:85} INFO - Job 50: Subtask evaluate_models
[2025-10-28, 00:41:56 UTC] {task_command.py:416} INFO - Running <TaskInstance: hiring_dynamic.evaluate_models scheduled__2024-10-05T15:00:00+00:00 [running]> on host d1f69dfe6243
[2025-10-28, 00:41:56 UTC] {taskinstance.py:1662} INFO - Exporting env vars: AIRFLOW_CTX_DAG_OWNER='airflow' AIRFLOW_CTX_DAG_ID='hiring_dynamic' AIRFLOW_CTX_TASK_ID='evaluate_models' AIRFLOW_CTX_EXECUTION_DATE='2024-10-05T15:00:00+00:00' AIRFLOW_CTX_TRY_NUMBER='1' AIRFLOW_CTX_DAG_RUN_ID='scheduled__2024-10-05T15:00:00+00:00'
[2025-10-28, 00:41:56 UTC] {logging_mixin.py:154} INFO - Evaluando 3 modelos en experimento 'dynamic'...
[2025-10-28, 00:41:56 UTC] {logging_mixin.py:154} INFO -   logistic_regression: Accuracy = 0.8400
[2025-10-28, 00:41:56 UTC] {logging_mixin.py:154} INFO -   gradient_boosting: Accuracy = 0.9200
[2025-10-28, 00:41:56 UTC] {logging_mixin.py:154} INFO -   random_forest: Accuracy = 0.9400
[2025-10-28, 00:41:56 UTC] {logging_mixin.py:154} INFO - 
✅ Mejor modelo (dynamic): random_forest
[2025-10-28, 00:41:56 UTC] {logging_mixin.py:154} INFO -    Accuracy: 0.9400
[2025-10-28, 00:41:56 UTC] {logging_mixin.py:154} INFO -    Guardado en: /opt/airflow/data/dynamic/2024-10-05/models/best_model.joblib
```
Y Para noviembre:

```yaml
*** Found local files:
***   * /opt/airflow/logs/dag_id=hiring_dynamic/run_id=scheduled__2024-11-05T15:00:00+00:00/task_id=evaluate_models/attempt=1.log
[2025-10-28, 00:42:28 UTC] {taskinstance.py:1159} INFO - Dependencies all met for dep_context=non-requeueable deps ti=<TaskInstance: hiring_dynamic.evaluate_models scheduled__2024-11-05T15:00:00+00:00 [queued]>
[2025-10-28, 00:42:28 UTC] {taskinstance.py:1159} INFO - Dependencies all met for dep_context=requeueable deps ti=<TaskInstance: hiring_dynamic.evaluate_models scheduled__2024-11-05T15:00:00+00:00 [queued]>
[2025-10-28, 00:42:28 UTC] {taskinstance.py:1361} INFO - Starting attempt 1 of 1
[2025-10-28, 00:42:28 UTC] {taskinstance.py:1382} INFO - Executing <Task(PythonOperator): evaluate_models> on 2024-11-05 15:00:00+00:00
[2025-10-28, 00:42:28 UTC] {standard_task_runner.py:57} INFO - Started process 1358 to run task
[2025-10-28, 00:42:28 UTC] {standard_task_runner.py:84} INFO - Running: ['airflow', 'tasks', 'run', 'hiring_dynamic', 'evaluate_models', 'scheduled__2024-11-05T15:00:00+00:00', '--job-id', '60', '--raw', '--subdir', 'DAGS_FOLDER/dag_dynamic.py', '--cfg-path', '/tmp/tmpidndc3bk']
[2025-10-28, 00:42:28 UTC] {standard_task_runner.py:85} INFO - Job 60: Subtask evaluate_models
[2025-10-28, 00:42:28 UTC] {task_command.py:416} INFO - Running <TaskInstance: hiring_dynamic.evaluate_models scheduled__2024-11-05T15:00:00+00:00 [running]> on host d1f69dfe6243
[2025-10-28, 00:42:28 UTC] {taskinstance.py:1662} INFO - Exporting env vars: AIRFLOW_CTX_DAG_OWNER='airflow' AIRFLOW_CTX_DAG_ID='hiring_dynamic' AIRFLOW_CTX_TASK_ID='evaluate_models' AIRFLOW_CTX_EXECUTION_DATE='2024-11-05T15:00:00+00:00' AIRFLOW_CTX_TRY_NUMBER='1' AIRFLOW_CTX_DAG_RUN_ID='scheduled__2024-11-05T15:00:00+00:00'
[2025-10-28, 00:42:28 UTC] {logging_mixin.py:154} INFO - Evaluando 3 modelos en experimento 'dynamic'...
[2025-10-28, 00:42:28 UTC] {logging_mixin.py:154} INFO -   logistic_regression: Accuracy = 0.8700
[2025-10-28, 00:42:28 UTC] {logging_mixin.py:154} INFO -   gradient_boosting: Accuracy = 0.9300
[2025-10-28, 00:42:28 UTC] {logging_mixin.py:154} INFO -   random_forest: Accuracy = 0.9467
[2025-10-28, 00:42:28 UTC] {logging_mixin.py:154} INFO - 
✅ Mejor modelo (dynamic): random_forest
[2025-10-28, 00:42:28 UTC] {logging_mixin.py:154} INFO -    Accuracy: 0.9467
[2025-10-28, 00:42:28 UTC] {logging_mixin.py:154} INFO -    Guardado en: /opt/airflow/data/dynamic/2024-11-05/models/best_model.joblib
```
Respondiendo las preguntas:
- ¿Cual es el accuracy de cada modelo en la ejecución de octubre? ¿Se obtienen los mismos resultados a partir de Noviembre?

El accuracy de cada modelo en la ejecución de octubre es:
  - logistic_regression: 0.8400
  - gradient_boosting: 0.9200
  - random_forest: 0.9400

El accuracy de cada modelo en la ejecución de noviembre es:
  - logistic_regression: 0.8700
  - gradient_boosting: 0.9300
  - random_forest: 0.9467

Se obtienen resultados similares, pero con ligeras mejoras en noviembre debido a la incorporación de nuevos datos.

- Analice como afectó el añadir datos a sus modelos mediante el desempeño del modelo y en costo computacional.

Se obtienen resultados similares, pero con ligeras mejoras en noviembre debido a la incorporación de nuevos datos. El costo computacional se ve reflejado en la task duration: 

(ver https://github.com/danmc899/MDS7202/blob/main/Entregables_Lab9/dags/Images/Costo%20dynamic.png si no carga la imagen)
<p align="center"><img src="dags/Images/Costo dynamic.png" width="1200"></p>:


Acá se observa que regresión logística tuvo un aumento significativo en su tiempo de ejecución, pasando de 0.46 segundos a 8.75 segundos, lo que da indicios de que no vale la pena complejizar el entrenamiento si los resultados variarán tan poco. El tiempo de entrenamiento para XGBoost y Random Forest son similares.

# Conclusión

Éxito!
<div align="center">
  <img src="https://miro.medium.com/v2/resize:fit:1000/1*PX8WVijZapo7EDrvGv9Inw.gif" width="500">
</div>
